In [100]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [101]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device use : {device}")

device use : cuda


In [102]:
df = pd.read_csv("/content/fashion-mnist_test.csv")
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
750,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8319,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3258,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9540,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
845,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [103]:
x = df.drop(columns=['label']).values
y = df['label'].values

In [104]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)

In [105]:
x_train = x_train/255.0
x_test = x_test/255.0

In [106]:
class sampledata(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features,dtype = torch.float32).reshape(-1,1,28,28)
    self.labels = torch.tensor(labels,dtype=torch.long)
  def __len__(self):
    return len(self.features)
  def __getitem__(self,idx):
    return self.features[idx],self.labels[idx]


In [107]:
train_dataset = sampledata(x_train,y_train)
test_dataset = sampledata(x_test,y_test)

In [108]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32,shuffle = False)

In [109]:
class myCNN(nn.Module):
  def __init__(self,input_dim):
    super().__init__()
    self.features = nn.Sequential(
         nn.Conv2d(input_dim,32,kernel_size=3,padding = 'same'),
         nn.ReLU(),
         nn.BatchNorm2d(32),
         nn.MaxPool2d(kernel_size=2,stride = 2),

         nn.Conv2d(32,64,kernel_size=3,padding = 'same'),
         nn.ReLU(),
         nn.BatchNorm2d(64),
         nn.MaxPool2d(kernel_size = 2,stride = 2)

     )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7 , 128),
        nn.ReLU(),
        nn.Dropout(p = 0.2),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p = 0.2),

        nn.Linear(64,10)
    )
  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [110]:
model = myCNN(1)
model.to(device)

myCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [111]:
learning_rate = 0.01
epochs = 25

In [112]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = learning_rate)

In [113]:
for epoch in range(epochs):
  for batch,labels in train_loader:
    batch = batch.to(device)
    labels = labels.to(device)
    output = model(batch)
    loss = loss_function(output,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print(f"num epoch{epoch+1} : loss : {loss.item()}")


num epoch1 : loss : 0.7519388198852539
num epoch2 : loss : 0.28578442335128784
num epoch3 : loss : 0.41069671511650085
num epoch4 : loss : 0.3113483488559723
num epoch5 : loss : 0.2423427551984787
num epoch6 : loss : 0.47141313552856445
num epoch7 : loss : 0.1540856659412384
num epoch8 : loss : 0.23001861572265625
num epoch9 : loss : 0.3511071503162384
num epoch10 : loss : 0.472095251083374
num epoch11 : loss : 0.1413956582546234
num epoch12 : loss : 0.16785353422164917
num epoch13 : loss : 0.17982661724090576
num epoch14 : loss : 0.6714087724685669
num epoch15 : loss : 0.281398206949234
num epoch16 : loss : 0.3749358057975769
num epoch17 : loss : 0.39548540115356445
num epoch18 : loss : 0.10731333494186401
num epoch19 : loss : 0.54150390625
num epoch20 : loss : 0.15107354521751404
num epoch21 : loss : 0.22220651805400848
num epoch22 : loss : 0.32651597261428833
num epoch23 : loss : 0.43004196882247925
num epoch24 : loss : 0.334541380405426
num epoch25 : loss : 1.812143325805664


In [114]:
model.eval()

myCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [115]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.851


In [116]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.898625
